# 第4课：Softmax 与分类任务

**学习目标：**
- 用 PyTorch 实现 Softmax 分类
- 对比 NumPy 和 PyTorch 的实现差异
- 完成端到端的分类推理

---

在 NumPy 教程的第4-5课，我们实现了 Softmax 和分类任务。本课用 PyTorch 重做一遍，你会发现代码简洁了很多。

## 4.1 PyTorch 的 Softmax

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 方法1：函数式 API
logits = torch.tensor([[2.0, 1.0, 0.1],
                        [0.5, 2.5, 0.3]])
probs = F.softmax(logits, dim=1)  # dim=1 表示沿类别维度
print("概率分布:")
print(probs)
print("每行之和:", probs.sum(dim=1))  # 应该都是 1.0

# 方法2：nn.Module
softmax = nn.Softmax(dim=1)
print("\nnn.Softmax:", softmax(logits))

## 4.2 构建分类网络

In [ ]:
class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU(),
            nn.Linear(16, 2)  # 输出2个类别的 logits
        )
    
    def forward(self, x):
        return self.network(x)  # 返回 logits
    
    def predict(self, x):
        """返回类别标签"""
        logits = self.forward(x)
        probs = F.softmax(logits, dim=1)
        return torch.argmax(probs, dim=1)  # 取概率最大的类别

model = Classifier()
print(model)

## 4.3 分类推理

未训练的网络输出是随机的（和 NumPy 版一样）：

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')
from pytorch.utils import create_data, plot_data

# 生成数据
data = create_data(500)
X = torch.tensor(data[:, :2], dtype=torch.float32)

# 预测
model.eval()  # 切换到评估模式
with torch.no_grad():
    predictions = model.predict(X).numpy()

# 可视化
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plot_data(data, "真实分类")

predicted_data = data.copy()
predicted_data[:, 2] = predictions
plt.subplot(1, 2, 2)
plot_data(predicted_data, "未训练网络预测")

plt.tight_layout()
plt.show()

## 4.4 argmax vs 人工分类

PyTorch 的 `argmax` 比 NumPy 手写的 `np.rint` 更通用：
- `np.rint`：只能处理2分类
- `argmax`：适用于任意数量的类别

In [ ]:
# 3分类示例
logits = torch.tensor([[2.0, 5.0, 1.0],
                        [3.0, 1.0, 4.0],
                        [1.0, 2.0, 3.0]])

probs = F.softmax(logits, dim=1)
predictions = torch.argmax(probs, dim=1)

print("概率:", probs)
print("预测类别:", predictions)  # tensor([1, 2, 2])

---

## 小结

- `F.softmax(logits, dim=1)` 一行搞定 Softmax
- `torch.argmax(probs, dim=1)` 取最大概率的类别
- `model.eval()` + `torch.no_grad()` 用于推理
- PyTorch 的分类代码比 NumPy 简洁得多

**下一课**我们将引入损失函数和优化器，完成真正的训练。